# Экзамен по дисциплине «Глубокое обучение»

## Тема: визуальный контроль качества дорожной инфраструктуры

**Цель экзамена** — проверить умение применять базовые концепции глубокого обучения на практике:
подготовить данные, обучить простую полносвязную и сверточную сеть, добавить регуляризацию и интерпретировать результат.

**Прикладной контекст.** Вы инженер команды системы «Умный город». Камера автомобиля снимает дорогу, и система должна определить тип дефекта покрытия. Для экзамена данные синтезируются прямо в ноутбуке —
не нужно ничего скачивать.

---

### Регламент

- Ноутбук рассчитан на запуск в Google Colab с GPU T4 (CPU тоже подойдет).

### Жесткие технические ограничения (нарушать нельзя)

| Параметр | Значение |
|----------|----------|
| Максимум эпох на одну модель | **10** |
| Максимум CNN-моделей в задании 3 | **3** (базовая + две модификации) |
| Размер батча | **64** (фиксированный) |
| Оптимизатор | **Adam, lr = 1e-3** (если не указано иное) |
| Размер изображений | **32×32, grayscale** |

Что не является целью задания (например, генерация датасета, train loop, графики кривых), дано в готовом виде. Ваша задача — **использовать** эти заготовки, а не переписывать их.

## Структура и баллы

| № | Задание | Баллы |
|---|---------|-------|
| 1 | Постановка задачи и аудит данных | **1** |
| 2 | Подготовка данных и baseline на простейшей сети (MLP) | **2** |
| 3 | Сверточная модель и анализ переобучения | **3** |
| 4 | Регуляризация и устойчивость модели | **2** |
| 5 | Сравнение моделей. Интерпретация и презентация результатов | **2** |
| | **Итого** | **10** |

**За что снимаются баллы:**
- пустые или однострочные выводы («модель обучена, accuracy = 0.87»);
- отсутствие сравнения там, где оно требуется;
- нарушение технических ограничений (>10 эпох, >3 CNN и т. п.);
- копирование кода без понимания.

> *Опциональный бонусный блок про AE/VAE для поиска аномалий вынесен в приложение и **не оценивается**.*

## Настройка окружения

In [ ]:
import math
import random
import warnings

import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Устройство: {DEVICE}")

warnings.filterwarnings("ignore")

# Фиксированные гиперпараметры — менять нельзя
BATCH_SIZE = 64
MAX_EPOCHS = 10
LEARNING_RATE = 1e-3
IMG_SIZE = 32
NUM_CLASSES = 4
CLASS_NAMES = ["ровное", "трещина", "выбоина", "заплатка"]


## Готовый код: генерация датасета

Ниже — **готовая** функция генерации синтетического датасета дорожных дефектов.
**Запустите ячейку и используйте результат — переписывать ее не нужно.**

Получаются:
- `X` — массив изображений `(N, 32, 32)`, значения в `[0, 1]`;
- `y` — массив меток `(N,)` со значениями 0..3;
- классы: `0 — ровное покрытие`, `1 — трещина`, `2 — выбоина`, `3 — заплатка`.

Всего **2000 изображений**, **сбалансированы** по классам (по 500 на класс).

In [ ]:
def generate_road_dataset(n_per_class=500, img_size=32, seed=SEED):
    """Синтетический датасет дорожных дефектов. Возвращает (X, y)."""
    rng = np.random.RandomState(seed)
    images, labels = [], []

    for cls in range(4):
        for _ in range(n_per_class):
            # фон: серое покрытие с шумом
            img = 0.5 + 0.05 * rng.randn(img_size, img_size)

            if cls == 0:  # ровное
                pass
            elif cls == 1:  # трещина — тонкая ломаная линия
                x0, y0 = rng.randint(0, img_size, size=2)
                angle = rng.uniform(0, 2 * np.pi)
                length = rng.randint(15, img_size)
                for t in range(length):
                    x = int(x0 + t * np.cos(angle))
                    y = int(y0 + t * np.sin(angle))
                    if 0 <= x < img_size and 0 <= y < img_size:
                        img[y, x] -= rng.uniform(0.25, 0.45)
            elif cls == 2:  # выбоина — округлое темное пятно
                cx, cy = rng.randint(6, img_size - 6, size=2)
                r = rng.randint(3, 7)
                yy, xx = np.ogrid[:img_size, :img_size]
                mask = (xx - cx) ** 2 + (yy - cy) ** 2 <= r ** 2
                img[mask] -= rng.uniform(0.3, 0.5)
            elif cls == 3:  # заплатка — прямоугольник
                w, h = rng.randint(8, 16, size=2)
                x0 = rng.randint(0, img_size - w)
                y0 = rng.randint(0, img_size - h)
                img[y0:y0 + h, x0:x0 + w] -= rng.uniform(0.15, 0.3)

            img = np.clip(img, 0.0, 1.0).astype(np.float32)
            images.append(img)
            labels.append(cls)

    X = np.stack(images)
    y = np.array(labels, dtype=np.int64)
    # перемешаем
    idx = rng.permutation(len(X))
    return X[idx], y[idx]


X, y = generate_road_dataset()
print(f"Сгенерировано: X={X.shape}, y={y.shape}")
print(f"Распределение по классам: {np.bincount(y)}")


## Готовый код: train loop и визуализация кривых

Ниже — **готовые** функции для обучения и визуализации, которыми вы будете пользоваться во всех заданиях.
Сигнатура и поведение фиксированы, ваша задача — только передавать в них модель и данные.

In [ ]:
def train_model(model, train_loader, val_loader, epochs=MAX_EPOCHS,
                lr=LEARNING_RATE, weight_decay=0.0, verbose=True):
    """Универсальный train loop. Возвращает словарь с историей метрик."""
    assert epochs <= MAX_EPOCHS, f"Запрещено больше {MAX_EPOCHS} эпох"
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(1, epochs + 1):
        # --- train ---
        model.train()
        tr_loss, tr_correct, tr_total = 0.0, 0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            tr_loss += loss.item() * xb.size(0)
            tr_correct += (logits.argmax(1) == yb).sum().item()
            tr_total += xb.size(0)

        # --- val ---
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                logits = model(xb)
                loss = criterion(logits, yb)
                val_loss += loss.item() * xb.size(0)
                val_correct += (logits.argmax(1) == yb).sum().item()
                val_total += xb.size(0)

        history["train_loss"].append(tr_loss / tr_total)
        history["val_loss"].append(val_loss / val_total)
        history["train_acc"].append(tr_correct / tr_total)
        history["val_acc"].append(val_correct / val_total)

        if verbose:
            print(f"epoch {epoch:2d} | "
                  f"train_loss={history['train_loss'][-1]:.4f} "
                  f"val_loss={history['val_loss'][-1]:.4f} | "
                  f"train_acc={history['train_acc'][-1]:.3f} "
                  f"val_acc={history['val_acc'][-1]:.3f}")

    return history


def plot_curves(history, title=""):
    """Кривые train/val loss и accuracy."""
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(history["train_loss"], label="train")
    ax[0].plot(history["val_loss"], label="val")
    ax[0].set_title(f"{title} loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(history["train_acc"], label="train")
    ax[1].plot(history["val_acc"], label="val")
    ax[1].set_title(f"{title} accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend()
    plt.tight_layout(); plt.show()


@torch.no_grad()
def evaluate(model, loader):
    """Возвращает (y_true, y_pred) для loader."""
    model.eval()
    ys, ps = [], []
    for xb, yb in loader:
        xb = xb.to(DEVICE)
        logits = model(xb)
        ps.append(logits.argmax(1).cpu().numpy())
        ys.append(yb.numpy())
    return np.concatenate(ys), np.concatenate(ps)


## Задание 1. Постановка задачи и аудит данных (1 балл)

### Зачем
Любой ML-проект начинается с понимания данных. Прежде чем строить модель, надо посмотреть,
что вообще приходит на вход и насколько классы отличимы.

### Что нужно сделать
1. **Краткая постановка задачи (3–5 предложений).** Опишите своими словами:
   - что является объектом классификации;
   - какие классы (ориентируйтесь на таблицу ниже);
   - почему ошибки могут стоить дорого в реальной эксплуатации.

2. **Визуализация данных.** Покажите по **3 примера каждого класса** (всего 12 картинок) с подписями.

3. **Баланс классов.** Постройте bar-chart распределения классов и одной фразой прокомментируйте,
   сбалансирована ли выборка.

> Подробный анализ статистик пикселей **не требуется** — достаточно картинок и баланса классов.

| Класс | Название |
|-------|----------|
| 0 | Ровное покрытие |
| 1 | Трещина |
| 2 | Выбоина |
| 3 | Заплатка |

In [ ]:
"""ВАШЕ РЕШЕНИЕ"""


**Вывод (2–4 предложения):**

_Опишите:_
- _насколько классы визуально различимы — легко ли их различить на глаз;_
- _какой класс, по вашему мнению, будет самым сложным для модели и почему._

## Задание 2. Подготовка данных и baseline на простейшей сети (2 балла)

### Зачем
Baseline — это нижняя граница качества. Если сложный CNN не бьет простую MLP, значит, где-то ошибка
(переобучение, неправильная нормализация, утечка данных).

### Что нужно сделать
1. **Разбейте train/val/test** в пропорции 70/15/15 со `stratify=y` и зафиксированным seed.

2. **Нормализация:** вычтите среднее и поделите на std, рассчитанные **только на train**.

3. **Создайте `DataLoader`** для train/val/test с `batch_size = BATCH_SIZE`.

4. **Постройте MLP-baseline** на «расплющенных» пикселях
   (вход 1024 → 2 скрытых слоя с ReLU → выход 4). Архитектуру выбирайте сами,
   но без излишеств: достаточно `Linear(1024, 128) → ReLU → Linear(128, 64) → ReLU → Linear(64, 4)`.

5. **Обучите** ровно `MAX_EPOCHS` эпох с помощью готового `train_model`. Постройте кривые через `plot_curves`.

6. **Получите метрики на тесте** через `evaluate` + `classification_report`.

### Вопросы для вывода (одно-два предложения на каждый)
- Почему MLP не оптимальна для изображений?
- Какие классы MLP путает чаще всего? (по `classification_report` или confusion matrix)

In [ ]:
"""ВАШЕ РЕШЕНИЕ"""


**Вывод (3–5 предложений):**

_Опишите:_
- _какой accuracy/F1 получился у MLP;_
- _ответы на два вопроса выше._

## Задание 3. Сверточная модель и анализ переобучения (3 балла)

### Зачем
CNN встраивает в архитектуру предположение о локальности и трансляционной инвариантности признаков
(край, угол, текстура — где бы они ни оказались на картинке). Цель задания — увидеть это на практике
и научиться читать кривые обучения.

### Что нужно сделать

**Ограничение: ровно три модели — базовая CNN и две ее модификации. Больше — нельзя.**

1. **Базовая CNN.** Минимальная архитектура (можно использовать как есть):
   ```
   Conv2d(1, 16, 3, padding=1) → ReLU → MaxPool(2)
   Conv2d(16, 32, 3, padding=1) → ReLU → MaxPool(2)
   Flatten → Linear(32*8*8, 64) → ReLU → Linear(64, 4)
   ```

2. **Две модификации на ваш выбор из списка ниже** (по одному изменению относительно базовой каждая,
   модификации должны различаться):
   - **глубже** — добавить еще один conv-блок;
   - **шире** — увеличить число фильтров вдвое;
   - **с BatchNorm** — добавить BatchNorm после сверток;
   - **другое ядро** — 5×5 вместо 3×3.

   Кратко обоснуйте свой выбор (одно предложение на каждую модификацию) **до** обучения.

3. **Обучите все три модели** через `train_model` (10 эпох, batch=64). Постройте кривые `plot_curves`
   для каждой и **визуально определите**, есть ли переобучение.

4. **Сводная таблица** (можно в виде markdown-таблицы или DataFrame):

   | Модель | # параметров | test accuracy | test F1-macro |
   |--------|--------------|---------------|---------------|
   | MLP (из задания 2) | ... | ... | ... |
   | CNN базовая | ... | ... | ... |
   | CNN модификация 1 | ... | ... | ... |
   | CNN модификация 2 | ... | ... | ... |

   *Подсказка для подсчета параметров:* `sum(p.numel() for p in model.parameters())`.

> Подсказка по переобучению. Если `val_loss` начинает расти, а `train_loss` продолжает падать —
> это переобучение. Запомните это для задания 4.

In [ ]:
"""ВАШЕ РЕШЕНИЕ"""


**Вывод (4–6 предложений):**

_Опишите:_
- _какая из трех моделей оказалась лучшей и за счет чего;_
- _какая из двух модификаций дала больший эффект и почему — исходя из природы данных;_
- _есть ли у какой-то из моделей переобучение и как вы это видите на кривых;_
- _оправдано ли усложнение архитектуры с точки зрения «качество vs число параметров»._

## Задание 4. Регуляризация и устойчивость модели (2 балла)

### Зачем
Регуляризация снижает переобучение и делает модель устойчивее к небольшим изменениям входа.
В этом задании вы добавляете регуляризацию к лучшей CNN из задания 3 и сравниваете было/стало.

### Что нужно сделать

1. Возьмите **лучшую CNN из задания 3** (если переобучения нет — все равно берите ее).

2. Добавьте **минимум два** из следующих механизмов (но не больше четырех):

   | Метод | Где применять |
   |-------|---------------|
   | **Dropout** (`p=0.3–0.5`) | после ReLU перед последним Linear |
   | **Weight decay (L2)** | параметр `weight_decay` у `train_model` (`1e-4` … `1e-3`) |
   | **Batch Normalization** | после conv-слоев (если еще не было) |
   | **Аугментация** (горизонтальный flip и/или небольшой шум) | в `__getitem__` своего `Dataset` или прямо в train loop |

3. **Обучите** регуляризованную модель (10 эпох, batch=64) и постройте кривые.

4. **Сравните «без регуляризации» vs «с регуляризацией»** в одной таблице:

   | Модель | train acc | val acc | test acc | gap (train − val) |
   |--------|-----------|---------|----------|-------------------|
   | CNN без регуляризации | ... | ... | ... | ... |
   | CNN с регуляризацией | ... | ... | ... | ... |

> Абляция (каждый компонент по отдельности) **не требуется** — достаточно сравнения было/стало.

### Один вопрос для вывода
- Какие аугментации **вредны** для задачи дорожных дефектов и почему?
  (Подумайте, например, про сильное размытие или вертикальный flip.)

In [ ]:
"""ВАШЕ РЕШЕНИЕ"""


**Вывод (3–5 предложений):**

_Опишите:_
- _что произошло с gap между train и val после регуляризации;_
- _какая комбинация методов сработала и почему;_
- _ответ на вопрос про вредные аугментации._

## Задание 5. Сравнение моделей. Интерпретация и презентация результатов (2 балла)

### Зачем
Метрика accuracy 0.87 сама по себе мало что говорит. Инженеру важно понимать,
**где** модель ошибается и **почему**, и уметь сравнить несколько моделей по понятным критериям.

### Что нужно сделать

1. **Сводная таблица всех моделей** проекта (MLP, CNN базовая, CNN модификация, CNN с регуляризацией):

   | Модель | # параметров | test accuracy | test F1-macro |
   |--------|--------------|---------------|---------------|
   | ... | ... | ... | ... |

2. **Confusion matrix лучшей модели** (нормированная по строкам, в %, с подписями классов).
   Выделите одной фразой: какая пара классов путается чаще всего.

3. **Примеры ошибок.** Покажите 3 изображения, на которых лучшая модель ошиблась с высокой
   уверенностью (`softmax > 0.8`). Подпишите для каждого: истинный класс, предсказанный класс, уверенность.

4. **Инженерные выводы (5–7 предложений).** Ответьте по пунктам:
   - какая модель и почему — итоговый выбор для продакшена;
   - какие систематические ошибки у этой модели;
   - что бы вы сделали следующим шагом для улучшения качества (1–2 идеи).

In [ ]:
"""ВАШЕ РЕШЕНИЕ"""


**Итоговый вывод проекта (5–7 предложений):**

_Опишите:_
- _какая модель в итоге выбрана и почему;_
- _где у нее системные слабости (по confusion matrix и примерам ошибок);_
- _ваши рекомендации по дальнейшему развитию системы._

---

## Приложение (не оценивается). AE и VAE для поиска аномалий

Этот блок **полностью опциональный** — он не влияет на оценку. Делайте, только если интересно
и есть свободное время. Можно ограничиться только AE — VAE добавляйте по желанию.

**Идея.** Классификатор не умеет говорить «я не знаю». Если на дороге появится новый тип покрытия
(например, брусчатка), CNN все равно уверенно отнесет его к одному из четырех классов.
Автоэнкодер, обученный только на норме, умеет ловить такое: у незнакомых объектов высокая
ошибка реконструкции.

### Часть A. AE (минимум)

1. Обучите простой сверточный AE **только на классе 0** (ровное покрытие).
   Архитектура: CNN-энкодер → латент 16–32 → CNN-декодер. Loss: MSE.
2. Посчитайте ошибку реконструкции на всех классах теста.
3. Постройте boxplot / гистограмму ошибки по классам — отделяется ли норма от дефектов?

### Часть B. VAE (по желанию, поверх части A)

1. Реализуйте **VAE** с тем же энкодером/декодером, но с `μ` и `log σ²` в латенте.
   Loss: `MSE + β · KL`, начните с маленького `β` (например, `1e-3`), иначе KL «задавит» реконструкцию.
2. Посчитайте ту же ошибку реконструкции по классам, что и для AE.
3. Сравните AE и VAE одной короткой таблицей или парой графиков:
   качество отделения нормы от дефектов (например, AUC ROC по ошибке реконструкции).

Подробных требований и оценки нет. Если что-то нестабильно обучается — это нормально для VAE
на маленьком латенте, не тратьте на это много времени.

In [ ]:
"""ОПЦИОНАЛЬНО — без оценки. Часть A: AE"""


In [ ]:
"""ОПЦИОНАЛЬНО — без оценки. Часть B: VAE (по желанию)"""
